# Step 2: Regularized Teacher Tree Selection

This notebook searches a small grid of regularized CART teacher trees and selects the simplest model whose validation RMSE remains near-optimal.

The selected teacher is the final predictive and explanatory model for the final decision-tree workflow.


In [ ]:
from pathlib import Path
import sys

def find_project_root() -> Path:
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "data" / "hdb_resale_final_modeling_dataset_2015_2025.csv").exists():
            return candidate
    raise FileNotFoundError("Could not find the project root from the current working directory.")

PROJECT_ROOT = find_project_root()
DT_V4_DIR = PROJECT_ROOT / "results" / "DT_models"
if str(DT_V4_DIR) not in sys.path:
    sys.path.insert(0, str(DT_V4_DIR))


In [ ]:
import json
import pandas as pd
from IPython.display import Image, display
from dt_modelsv4_utils import JSON_PATH

summary = json.loads(JSON_PATH.read_text(encoding="utf-8"))

teacher_grid = pd.DataFrame(
    [
        {
            "max_depth": row["config"]["max_depth"],
            "min_samples_leaf": row["config"]["min_samples_leaf"],
            "min_samples_split": row["config"]["min_samples_split"],
            "depth": row["depth"],
            "leaf_count": row["leaf_count"],
            "validation_rmse": row["validation"]["rmse"],
            "validation_r2": row["validation"]["r2"],
            "test_rmse": row["test"]["rmse"],
            "test_r2": row["test"]["r2"],
        }
        for row in summary["step2_teacher_grid"]
    ]
)
display(teacher_grid.round(4))

teacher_importance = pd.DataFrame(summary["step2_selected_teacher"]["feature_importances"])
display(teacher_importance.head(12).round(4))

print("Selected teacher root:", summary["step2_selected_teacher"]["root_summary"])

display(Image(filename=summary["artifacts"]["teacher_importance_png_path"]))
display(Image(filename=summary["artifacts"]["teacher_png_path"]))
